<a href="https://colab.research.google.com/github/lestermartin/Security_Labs/blob/master/workshops/ibis-pipelines/IbisWithStarburstGalaxy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Building Data Pipelines with Ibis and Starburst

This notebook represents the hands-on portion of the [Building Data Pipelines with Ibis and Starburst](https://github.com/starburstdata/devrel/tree/main/workshops/ibis-pipelines) webinar.



## Set YOUR env properties

The examples in the remainder of this notebook where build while using [Starburst Galaxy](https://www.starburst.io/starburst-galaxy/) (a super fast way to get going with Trino) using it's simple username/password authentication.  

To obtain the host name and your full user name, log into YOUR [Starburst Galaxy](https://www.starburst.io/starburst-galaxy/) tenant (if you don't have one, you can [sign up for LOTS of FREE credits](https://www.starburst.io/free-trial/)) and follow these steps.

1. From the *left-hand nav*, click on **Admin** > **Clusters**
2. From the **Clusters** list, scroll to the far right of the cluster you will be using and click on the *vertical elipsis* icon
3. Select **Partner connect** from the *contextual menu* that surfaces
4. Scroll down to the **Drivers & Clients** section and click on the Ibis tile
5. Use the **User** and **Host** properties present (plus your password) in the next cell

In [ ]:
import getpass

# grab credentials from the notebook user to be used when making a connection
my_host = input("Host name")
my_username = input("User name")
my_password = getpass.getpass("Password")

## Exploring Ibis

Python DataFrame API as described at the [Ibis project website](https://ibis-project.org/).

NOTE: Ibis can run against many different SQL engines, not just Trino.

### Env setup

In [ ]:
# install Ibis

%pip install trino
%pip install 'ibis-framework[trino]'
%pip install pystarburst

In [ ]:
# boiler-plate code for setup

import os
import ibis
from trino.auth import BasicAuthentication

ibis.options.interactive = True

user = my_username
trino_auth_obj = BasicAuthentication(my_username, my_password)
host = my_host
port = "443"
http_scheme = "https"
catalog = "mycloud" #"tpch"
schema = "book_testing" #"tiny"

con = ibis.trino.connect(
    user=user, auth=trino_auth_obj, host=host, port=port, http_scheme=http_scheme, database=catalog, schema=schema
)

print('\n Make sure the phrase ** CONNECTION IS GOOD ** displays \n')
con.sql("select '** CONNECTION IS GOOD **' as conn_check")

### Interactive exploration

You could slide into your problem solving by take baby steps to see what each bit is doing for you.

Note: This code was originally published in  [ibis & trino (dataframe api part deux)](https://lestermartin.blog/2023/10/27/ibis-trino-dataframe-api-part-deux/).  

In [ ]:
# select a full table

custDF = con.table("customer", database=("tpch", "tiny"))
custDF[0:10]

In [ ]:
# use projection

projectedDF = custDF.select("name", "acctbal", "nationkey")
projectedDF[0:10]

In [ ]:
# apply a predicate

filteredDF = projectedDF.filter(projectedDF["acctbal"] > 9900.0)
filteredDF[0:100]

In [ ]:
# select a second table

nationDF = con.table("nation", database=("tpch", "tiny")) \
            .drop("regionkey", "comment") \
            .rename(
                dict(
                    nation_name="name",
                    n_nationkey="nationkey"
                )
            )
nationDF[0:10]

In [ ]:
# join the tables

joinedDF = filteredDF.join(nationDF,
    filteredDF.nationkey == nationDF.n_nationkey)
joinedDF[0:10]

In [ ]:
# project the joined results

projectedJoinDF = joinedDF.drop("nationkey", "n_nationkey")
projectedJoinDF[0:10]

In [ ]:
# add sorting

orderedDF = projectedJoinDF.order_by([ibis.desc("acctbal")])
orderedDF[0:10]

#### Leverage lazy execution

The baby steps above were great for figuring out your logic, but if you focus only getting the final answer, all of the DataFrame transformations will simply wait until you have a final I/O action triggered.

In [ ]:
# put it all together

nationDF = con.table("nation", database=("tpch", "tiny")) \
            .drop("regionkey", "comment") \
            .rename(
                dict(
                    nation_name="name",
                    n_nationkey="nationkey"
                )
            )

custDF = con.table("customer", database=("tpch", "tiny")) \
            .select("name", "acctbal", "nationkey") \
            .filter(projectedDF["acctbal"] > 9900.0)

apiSQL = custDF.join(nationDF,
    custDF.nationkey == nationDF.n_nationkey) \
            .drop("nationkey", "n_nationkey") \
            .order_by([ibis.desc("acctbal")])

apiSQL[0:10]

#### SQL is available, too

Ibis strives to make their DataFrame API be the only (or at least primary) way you run your data operations, it sometimes it just might be better to write the SQL to get a different result and/or process to run.

NOTE: There could be a performance difference as Ibis is not doing anything with the SQL you write -- it just submits it like it is. Additionally, this approach hinders the optionality promise of Ibis due to SQL nuances among different SQL engines.

In [ ]:
# or... just run some SQL

sqlDF = con.sql("""
         SELECT c.name, c.acctbal, n.name AS nation_name
           FROM tpch.tiny.customer c
           JOIN tpch.tiny.nation n
             ON c.nationkey = n.nationkey
          WHERE c.acctbal > 9900.0
          ORDER BY c.acctbal DESC
""")
sqlDF[0:10]

### Richer examples

#### Example 1 (joining 3 tables)

Description: Aggregate total customer acctbal by region name

Tables: tpch.tiny.customer, tpch.tiny.nation, tpch.tiny.region

In [ ]:
nation = con.table("nation", database=f"tpch.tiny")
region = con.table("region", database=f"tpch.tiny")
customer = con.table("customer", database=f"tpch.tiny")

# nation >< region, n.regionkey = r.regionkey
nr = nation.join(region, nation["regionkey"] == region["regionkey"]).select(
    nation["nationkey"], region["name"].name("region_name")
)

# customer >< (nation, region) nr, c.nationkey = nr.nationkey
agg = (
    customer.join(nr, customer["nationkey"] == nr["nationkey"])
            .group_by("region_name")
            .aggregate(total_acctbal=customer["acctbal"].sum())
            .order_by(ibis.desc("total_acctbal"))
)

agg[0:10]

#### Example 2 (windowing example)

For each nation, get top N customers by acctbal

Tables: tpch.tiny.customer, tpch.tiny.nation

In [ ]:
expr = con.sql("""
        SELECT
          c.custkey,
          c.name,
          c.acctbal,
          n.name AS nation_name,
          row_number() OVER (
            PARTITION BY
              n.name
            ORDER BY
              c.acctbal DESC
          ) AS rn
        FROM
          tpch.tiny.customer c
          JOIN tpch.tiny.nation n ON c.nationkey = n.nationkey
        WHERE
          c.acctbal > 8000.0
""")
top_x = expr.filter(expr.rn == 10).order_by([ibis.desc("acctbal"), "nation_name"])

top_x[0:10]

## Real-world example

Let's use the Burst Bank customer table as if it is our raw source already ingested. We can focus on doing some limited quality checks, and transformations, as well as a precalculated aggregates table.

### Explore the data

Notice how the optional `database=("<catalog>", "<schema>")` parameter allows you to select something different than was configured in the `con` object.

Seems a good idea to always being this specific; especially for code to be shared and/or productionalized.

Let's start with the `customer` table.

In [ ]:

bank_customer = con.table("customer", database=("sample", "burstbank"))
bank_customer[0:20]

#### Addresses

A quick glance suggests there are US and CA addresses which already (further) suggests that `state` is actually US-oriented.

Let's see if the country/state combos seem to make sense.

In [ ]:
state_groups = bank_customer.group_by(["country", "state"]).aggregate(
    nbr_custs = bank_customer.custkey.count(),
    avg_fico  = bank_customer.fico.mean()
).order_by("country", "state")   #ibis.desc("nbr_custs"),)

state_groups.to_pandas().head(2000)


#print(state_groups.to_pandas)
#state_groups[0:200]



In [ ]:
t = conn.table("orders")

result = (
    t.group_by(["region", "product_category"])
    .aggregate(
        total_orders   = t.order_id.count(),
        avg_order_value= t.order_value.mean(),
        total_revenue  = t.order_value.sum(),
        unique_customers = t.customer_id.nunique(),
    )
    .order_by(ibis.desc("total_orders"))
)

print(result)

In [ ]:
result = (
    t.filter(t.order_date.year() == 2025)          # WHERE  — filter before grouping
    .group_by(["region", "product_category"])
    .aggregate(
        total_orders    = t.order_id.count(),
        avg_order_value = t.order_value.mean(),
    )
    .filter(ibis._.total_orders > 100)              # HAVING — filter after grouping
    .order_by(ibis.desc("avg_order_value"))
)

#### Phone numbers

The `phone` data looks to be all over the place. Let's install a phone number checking library and do a simple hard-coded test to make sure it works.

In [ ]:
%pip install phonenumbers

In [ ]:
import phonenumbers

def standardize_phone(raw: str, region: str = "US") -> str:
    parsed = phonenumbers.parse(raw, region)

    if not phonenumbers.is_valid_number(parsed):
        raise ValueError(f"Invalid phone number: {raw}")

    return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164)


# Examples
numbers = [
    "2025551234",
    "(202) 555-1234",
    "(202) 555-1234 ext. 103", # notice how extensions get lost
    "202-555-1234x103",
    "202.555.1234",
    "+1 202 555 1234",
    "1-202-555-1234",
]

for n in numbers:
    print(f"{n:25} -> {standardize_phone(n)}")

Pull out the phone number column from the dataframe and run the values against the generator to see what kind of shape this data is in.

In [ ]:
import pandas as pd
import phonenumbers

# quick wrapper around the phonenumbers library
def standardize_phone(raw: str, region: str = "US") -> str:
    try:
        parsed = phonenumbers.parse(str(raw), region)
        if not phonenumbers.is_valid_number(parsed):
            return None  # or return raw to keep original
        return phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164)
    except phonenumbers.NumberParseException:
        return None  # unparseable values return None

# Trino does NOT support this kind of basic Python UDF (needs SQL)
phone_pdf = bank_customer.select("phone").to_pandas()

# With the pandas DF create a new column for the cleaned up values
phone_pdf["phone_std"] = phone_pdf["phone"].apply(standardize_phone)
phone_pdf.head(1500)

Most values are not getting standardizing, but it isn't as bad as it seems as this is a generated list of numbers and most actually are invalid for one reason or another.

We did find out from the earlier test of `phonenumbers` that the extensions get tossed away.

We determined:

1. We cannot reliably perform standardization with the bad data we have at this point.
1. We need to go upstream to enhance the data collection application to be more rigorous than just capturing a string.

IF the data was in good enough shape to standardize a high-percentage of the phone numbers, we could include that as an additional column on the silver table that could be created from this bronze dataset.



In [ ]:
con.create_table("customers_clean2", phone_pdf, overwrite=True, database=("mycloud", "book_testing"))
print("Table created!")

In [ ]:
# TRASH THE CELLS ON THIS CSV BUNNY TRAIL ONE

import io
import zipfile
import urllib.request

def download_and_unzip(url: str, extract_to: str = ".") -> None:
    with urllib.request.urlopen(url) as response:
        data = response.read()

    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(extract_to)
        print(f"Extracted {len(zf.namelist())} files to '{extract_to}':")
        for name in zf.namelist():
            print(f"  {name}")



url = "https://s3.amazonaws.com/hubway-data/202501-bluebikes-tripdata.zip"
download_and_unzip(url, extract_to="output")

In [ ]:
ls -al output/*.csv

In [ ]:
raw_data = con.read_csv("output/202501-bluebikes-tripdata.csv")
raw_data[0:10]